In [ ]:
#!/usr/bin/env python3
"""
   Model C v2 — ConvNeXt V2 Large · Dark Matter Substructure Classification  
   Best of Model A + Best of Model B  (budget corrected from v1 logs)        
                                                                             
   From Model A  : 3-stage training · LLRD (12 groups) · EMA · BF16         
                   dataset norm stats · albumentations · Mixup α=0.4         
   From Model B  : MLP head (Linear→LN→GELU→Dropout→Linear) · TTAx8         
                                                                              
   v1 failure analysis (from logs):                                         
     S1 ended at 0.6387 (ep7), still climbing ~+0.009/ep. MLP head has     
     1187K params vs 4.6K single Linear; needs more warmup. Fixed: 7→12 ep  
     S2 hit cap at 0.9406 (ep30), gaining +0.005/ep, never plateaued.      
     Fixed: 30→50 max, patience 8→12                                         
     S3 hit cap at 0.9779 (ep15), gaining +0.001/ep, never plateaued.      
     Fixed: 15→30 max, patience 6→10                                         
                                                                            
   Budget : S1=12 fixed · S2=50 max (pat 12) · S3=30 max (pat 10)           
            92 max · ~70-80 expected · ES exits when genuinely saturated      
   Expected: >0.993 no TTA · >0.996 TTAx8                                   

"""

import os, math, random, gc, warnings
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import timm
from timm.utils import ModelEma

from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
warnings.filterwarnings('ignore')


# CONFIG
class CFG:
    # Paths
    DATA_ROOT  = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset"
    OUTPUT_DIR = "/kaggle/working/modelC_v2"

    # Model
    MODEL_NAME      = "convnextv2_large.fcmae_ft_in22k_in1k_384"
    NUM_CLASSES     = 3
    IMG_SIZE        = 224
    DROP_PATH       = 0.1

    # MLP Head 
    # Linear(1536→768) → LayerNorm → GELU → Dropout(0.3) → Linear(768→3)
    # Has 1187K params — needs longer S1 warmup than single Linear (4.6K)
    HEAD_HIDDEN_RATIO = 0.5
    HEAD_DROPOUT      = 0.3
    HEAD_INIT_SCALE   = 0.001   # near-zero init on final projection only

    # Stage epochs & early stopping
    #
    # S1 fix: 7→12 epochs
    #   v1 logs: val AUC at ep7 = 0.6387, still gaining ~+0.009/ep.
    #   Model A's single Linear (4.6K params) plateaued by ep10 at 0.6874.
    #   MLP head (1187K params) needs ~12 epochs to reach an equivalent
    #   starting point for backbone gradients.
    #
    # S2 fix: 30max/pat8 → 50max/pat12
    #   v1 logs: val AUC at ep30 = 0.9406, gaining +0.005/ep, never triggered ES.
    #   Model A reached 0.9753 by ep40. The weaker S1 start means S2 has
    #   more ground to cover. Patience 12 = ~1.3 epochs of noise tolerance
    #   at the typical +0.001/ep convergence rate near the ceiling.
    #
    # S3 fix: 15max/pat6 → 30max/pat10
    #   v1 logs: val AUC at ep15 = 0.9779, gaining +0.001/ep, never triggered ES.
    #   Model A's S3 gained +0.0178 over 40 epochs. We give 30 epochs with
    #   patience 10 so ES fires when it genuinely plateaus (~5 epochs stale
    #   at +0.0002/ep noise level), not because we ran out of budget.
    #
    STAGE1_EPOCHS   = 12
    STAGE2_EPOCHS   = 50
    STAGE2_PATIENCE = 12
    STAGE3_EPOCHS   = 30
    STAGE3_PATIENCE = 10

    # DataLoader 
    BATCH_SIZE   = 128
    NUM_WORKERS  = 4
    VAL_SPLIT    = 0.10

    #  Optimiser 
    BASE_LR      = 6.25e-4   # effective = 6.25e-4 × 128/256 = 3.125e-4
    WEIGHT_DECAY = 0.05
    LAYER_DECAY  = 0.7
    MIN_LR       = 1e-6

    #  Stage 3 LR 
    S3_LR = 3.125e-5          # 10× lower than S2 effective LR

    # Scheduler warmup 
    S1_WARMUP = 2             # 2-ep warmup for 12-ep S1 (same ratio as Model A)
    S2_WARMUP = 5             # 5-ep warmup for 50-ep S2 (same as Model A S2)

    #  Regularisation 
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA     = 0.4

    #  EMA
    EMA_DECAY = 0.9999

    #  Precision 
    USE_BF16  = True
    GRAD_CLIP = 1.0

    #  TTA 
    TTA_N_VIEWS = 8

    #  Dataset metadata 
    CLASS_NAMES = ['no_sub', 'subhalo', 'vortex']
    CLASS_DIRS  = {'no_sub': 'no', 'subhalo': 'sphere', 'vortex': 'vort'}
    PIXEL_MEAN  = 0.0615      # dataset statistics — NOT ImageNet
    PIXEL_STD   = 0.1152
    SEED        = 42


# REPRODUCIBILITY
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# EARLY STOPPING
class EarlyStopping:
    """
    Stops when val AUC shows no improvement for `patience` consecutive epochs.
    Counter is printed each epoch so you can see how close ES is to triggering.
    """
    def __init__(self, patience: int, label: str = ''):
        self.patience  = patience
        self.label     = label
        self.best      = -np.inf
        self.counter   = 0
        self.triggered = False

    def step(self, metric: float) -> bool:
        """Returns True if this epoch improved the best metric."""
        if metric > self.best:
            self.best    = metric
            self.counter = 0
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.triggered = True
        return False

    @property
    def status(self):
        if self.counter == 0:
            return ''
        return f'  [ES {self.counter}/{self.patience}]'


# DATASET
class LensDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        img = np.load(self.file_paths[idx]).astype(np.float32)
        if img.ndim == 3: img = img[0]           # (1,150,150) → (150,150)
        img_u8 = (img * 255).clip(0, 255).astype(np.uint8)
        if self.transform:
            return self.transform(image=img_u8)['image'], self.labels[idx]
        return torch.from_numpy(img_u8[None]).float() / 255.0, self.labels[idx]


def auto_discover_root(base='/kaggle/input') -> Path:
    expected = set(CFG.CLASS_DIRS.values())
    for dirpath, dirnames, _ in os.walk(base):
        p = Path(dirpath)
        if {'train', 'val'}.issubset(set(dirnames)):
            train_p = p / 'train'
            if train_p.exists():
                found = {d.name for d in train_p.iterdir() if d.is_dir()}
                if expected & found:
                    print(f"  ✓ Auto-discovered DATA_ROOT = {p}")
                    return p
    raise FileNotFoundError("Could not auto-discover dataset root.")


def build_file_list(root: Path, split: str):
    paths, labels = [], []
    for i, cls in enumerate(CFG.CLASS_NAMES):
        cls_dir   = root / split / CFG.CLASS_DIRS[cls]
        npy_files = sorted(cls_dir.glob("*.npy")) + sorted(cls_dir.glob("*.NPY"))
        if not npy_files:
            raise FileNotFoundError(f"No .npy files in {cls_dir}")
        paths.extend(str(p) for p in npy_files)
        labels.extend([i] * len(npy_files))
        print(f"  [{split}/{cls}]  {len(npy_files):,} images  ← {cls_dir}")
    return paths, labels


# AUGMENTATIONS
def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.9, border_mode=0, value=0),
        A.RandomResizedCrop(
            size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
            scale=(0.90, 1.00), ratio=(0.95, 1.05),
            interpolation=2, p=0.5,
        ),
        A.GaussNoise(var_limit=(0.65, 2.60), p=0.35),
        A.CoarseDropout(
            max_holes=4, max_height=18, max_width=18,
            min_holes=1, min_height=8,  min_width=8,
            fill_value=0, p=0.20,
        ),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_tta_transforms():
    """
    Label-preserving augmentations for TTA.
    Exploits 360° rotational symmetry of gravitational lenses.
    A new random combination of flip + rotation is applied per view.
    """
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=1.0, border_mode=0, value=0),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


# HELPERS
def replicate_channels(x): return x.repeat(1, 3, 1, 1)

def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# MLP HEAD
class MLPHead(nn.Module):
    """
    2-layer MLP head replacing timm's default single Linear classifier.

    Linear(1536 → 768) → LayerNorm(768) → GELU → Dropout(0.3) → Linear(768 → 3)

    Benefits over single Linear(1536→3):
      - LayerNorm re-scales the 768-dim space, stabilising gradients from
        heterogeneously-scaled backbone features (observed in FCMAE models)
      - Non-linear boundary via GELU — critical for subhalo/no_sub separation
        which is not linearly separable in the raw 1536-dim feature space
      - Dropout(0.3) adds regularisation in the intermediate space,
        complementing weight decay on the backbone parameters
    """
    def __init__(self, in_features: int, num_classes: int,
                 hidden_ratio: float = 0.5, dropout: float = 0.3):
        super().__init__()
        hidden = int(in_features * hidden_ratio)   # 1536 × 0.5 = 768
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.LayerNorm(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# MODEL BUILD
def build_model() -> nn.Module:
    model = timm.create_model(
        CFG.MODEL_NAME, pretrained=True,
        num_classes=CFG.NUM_CLASSES,
        drop_path_rate=CFG.DROP_PATH,
        in_chans=3,
    )

    # Swap timm's Linear head for MLPHead
    # timm ConvNeXt V2: model.head contains (norm → global_pool → fc)
    # model.head.fc = Linear(1536 → num_classes) — we replace fc only,
    # keeping the norm and global pool intact.
    in_features = model.head.fc.in_features        # 1536
    model.head.fc = MLPHead(
        in_features  = in_features,
        num_classes  = CFG.NUM_CLASSES,
        hidden_ratio = CFG.HEAD_HIDDEN_RATIO,
        dropout      = CFG.HEAD_DROPOUT,
    )

    # Near-zero init on final projection layer only.
    # The intermediate Linear uses default init so it can explore freely.
    final_linear = model.head.fc.net[-1]
    nn.init.trunc_normal_(final_linear.weight, std=0.02 * CFG.HEAD_INIT_SCALE)
    nn.init.constant_(final_linear.bias, 0)

    # Gradient checkpointing is essential: 197M params + MLP head + EMA
    # would otherwise OOM at batch=128 on H100 80GB during full fine-tune
    model.set_grad_checkpointing(enable=True)

    total    = sum(p.numel() for p in model.parameters())
    head_p   = sum(p.numel() for p in model.head.parameters())
    backbone = total - head_p
    print(f"  Model    : {CFG.MODEL_NAME}")
    print(f"  Backbone : {backbone/1e6:.1f}M params")
    print(f"  MLP Head : {head_p/1e3:.1f}K params  "
          f"(1536→768→LayerNorm→GELU→Dropout(0.3)→3)")
    print(f"  Total    : {total/1e6:.1f}M params  |  grad_ckpt=ON")
    return model


# LLRD PARAM GROUPS
def get_llrd_param_groups(model: nn.Module, base_lr: float) -> list:
    """
    12 param groups: wd + no-wd split per layer prefix.
    Depth order (shallow→deep): stem → stages.0-3 → head
    Head always gets full base_lr; each level deeper gets × LAYER_DECAY.

    'grn' in no_wd: ConvNeXt V2's Global Response Normalisation layers
    must not receive weight decay (they have learnable scale/bias params).
    """
    d     = CFG.LAYER_DECAY
    no_wd = {'bias', 'norm', 'bn', 'ln', 'gamma', 'beta',
             'LayerNorm', 'grn', 'scale'}

    layer_map = {
        'head':     1.0,
        'norm_pre': d ** 1,
        'stages.3': d ** 1,
        'stages.2': d ** 2,
        'stages.1': d ** 3,
        'stages.0': d ** 4,
        'stem':     d ** 5,
    }

    groups, assigned = [], set()

    def skip_wd(name): return any(k in name for k in no_wd)

    for prefix, scale in layer_map.items():
        wd_p, no_wd_p = [], []
        for name, param in model.named_parameters():
            if not param.requires_grad or name in assigned: continue
            if not name.startswith(prefix): continue
            assigned.add(name)
            (no_wd_p if skip_wd(name) else wd_p).append(param)
        if wd_p:
            groups.append({'params': wd_p,    'lr': base_lr * scale,
                           'weight_decay': CFG.WEIGHT_DECAY})
        if no_wd_p:
            groups.append({'params': no_wd_p, 'lr': base_lr * scale,
                           'weight_decay': 0.0})

    # Remaining params (e.g. BatchNorm in stem) → deepest decay level
    remaining = [(n, p) for n, p in model.named_parameters()
                 if p.requires_grad and n not in assigned]
    if remaining:
        groups.append({'params': [p for _, p in remaining],
                       'lr': base_lr * d**6, 'weight_decay': CFG.WEIGHT_DECAY})

    total_p = sum(p.numel() for g in groups for p in g['params'])
    print(f"  LLRD  : {len(groups)} groups | base_lr={base_lr:.2e} "
          f"| decay={d} | {total_p/1e6:.1f}M params")
    return groups


# SCHEDULER
def cosine_with_warmup(optimizer, warmup_epochs: int, total_epochs: int,
                        min_lr_ratio: float = 0.01):
    """Linear warmup then cosine decay to min_lr_ratio × base_lr."""
    def _fn(ep):
        if ep < warmup_epochs:
            return max(1e-6, (ep + 1) / warmup_epochs)
        prog = (ep - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr_ratio + 0.5*(1-min_lr_ratio)*(1+math.cos(math.pi*prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _fn)


# TRAIN ONE EPOCH
def train_one_epoch(model, loader, optimizer, criterion, scheduler,
                    ema, device, label: str) -> float:
    model.train()
    dtype  = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    losses = []
    pbar   = tqdm(loader, desc=label, leave=True)
    for imgs, labels in pbar:
        imgs   = replicate_channels(imgs).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG.MIXUP_ALPHA)
        with torch.autocast('cuda', dtype=dtype):
            logits = model(imgs)
            loss   = mixup_loss(criterion, logits, y_a, y_b, lam)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()
        ema.update(model)
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'lr':   f'{optimizer.param_groups[0]["lr"]:.2e}'})
    scheduler.step()
    return float(np.mean(losses))


# EVALUATE  (single-pass, used during training loop)
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dtype = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    all_probs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc='Eval', leave=False):
        imgs = replicate_channels(imgs).to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=dtype):
            probs = F.softmax(model(imgs), dim=-1)
        all_probs.append(probs.cpu().float().numpy())
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    macro  = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    per_cls = {cls: roc_auc_score((labels==i).astype(int), probs[:,i])
               for i, cls in enumerate(CFG.CLASS_NAMES)}
    return macro, per_cls, probs, labels


# TTA EVALUATE  (8 augmented views averaged at inference)
@torch.no_grad()
def evaluate_tta(model, file_paths, labels_list, device, n_views: int = 8):
    """
    Run n_views independent forward passes, each with a fresh random
    augmentation (different flip/rotation seed per view). Average the
    softmax probability vectors then renormalise.

    Why this works: Gravitational lenses are rotationally symmetric.
    Averaging predictions across 8 random orientations reduces variance
    in the probability estimate, directly improving ranking quality (AUC).
    """
    model.eval()
    dtype      = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    n_samples  = len(file_paths)
    prob_accum = np.zeros((n_samples, CFG.NUM_CLASSES), dtype=np.float32)

    print(f"\n  TTA inference ({n_views} views) ...")
    for view in range(n_views):
        loader = DataLoader(
            LensDataset(file_paths, labels_list, get_tta_transforms()),
            batch_size  = CFG.BATCH_SIZE * 2,
            shuffle     = False,
            num_workers = CFG.NUM_WORKERS,
            pin_memory  = True,
        )
        view_probs = []
        for imgs, _ in tqdm(loader, desc=f'  view {view+1}/{n_views}',
                            leave=False):
            imgs = replicate_channels(imgs).to(device, non_blocking=True)
            with torch.autocast('cuda', dtype=dtype):
                probs = F.softmax(model(imgs), dim=-1)
            view_probs.append(probs.cpu().float().numpy())
        prob_accum += np.concatenate(view_probs)

    # Average then renormalise — guards against any fp32 accumulation drift
    prob_accum  /= n_views
    prob_accum   = prob_accum.astype(np.float64)
    prob_accum  /= prob_accum.sum(axis=1, keepdims=True)

    labels  = np.array(labels_list)
    macro   = roc_auc_score(labels, prob_accum, multi_class='ovr', average='macro')
    per_cls = {cls: roc_auc_score((labels==i).astype(int), prob_accum[:,i])
               for i, cls in enumerate(CFG.CLASS_NAMES)}
    return macro, per_cls, prob_accum, labels


# PLOTS
def plot_roc_curves(labels, probs, save_path, title="Test Set"):
    COLORS = ['#e74c3c', '#2ecc71', '#3498db']
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'ROC Curves — {title}', fontsize=14, fontweight='bold')

    ax, per_aucs = axes[0], []
    for i, (cls, col) in enumerate(zip(CFG.CLASS_NAMES, COLORS)):
        binary      = (labels == i).astype(int)
        fpr, tpr, _ = roc_curve(binary, probs[:, i])
        auc_val     = roc_auc_score(binary, probs[:, i])
        per_aucs.append(auc_val)
        ax.plot(fpr, tpr, color=col, lw=2.2,
                label=f'{cls}  (AUC = {auc_val:.4f})')
    ax.plot([0,1],[0,1],'k--',lw=1,label='Random')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('One-vs-Rest ROC per Class')
    ax.legend(loc='lower right'); ax.grid(alpha=0.25)
    ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

    macro = np.mean(per_aucs)
    ax2   = axes[1]
    bars  = ax2.bar(CFG.CLASS_NAMES, per_aucs, color=COLORS,
                    alpha=0.85, edgecolor='white', linewidth=1.2)
    ax2.axhline(macro, color='gold', lw=2, ls='--',
                label=f'Macro = {macro:.4f}')
    ax2.set_ylim(max(0.5, min(per_aucs)-0.03), 1.005)
    ax2.set_ylabel('AUC'); ax2.set_title('Per-Class AUC Summary')
    ax2.legend(); ax2.grid(axis='y', alpha=0.25)
    for bar, val in zip(bars, per_aucs):
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                 f'{val:.4f}', ha='center', va='bottom',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ROC plot → {save_path}")


def plot_history(history, save_path, s1_ep, s2_ep, s3_ep):
    total = len(history['train_loss'])
    eps   = range(1, total + 1)
    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax1.plot(eps, history['train_loss'], color='#e74c3c', lw=2, label='Train Loss')

    s1_end = s1_ep
    s2_end = s1_ep + s2_ep
    ax1.axvspan(1,        s1_end+0.5, alpha=0.04, color='blue',   label='S1 (head)')
    ax1.axvspan(s1_end+0.5, s2_end+0.5, alpha=0.04, color='green', label='S2 (LLRD)')
    ax1.axvspan(s2_end+0.5, total+0.5, alpha=0.04, color='orange', label='S3 (polish)')
    ax1.axvline(s1_end+0.5, color='gray',   ls=':', lw=1.5)
    ax1.axvline(s2_end+0.5, color='orange', ls=':', lw=1.5)

    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss', color='#e74c3c')
    ax2 = ax1.twinx()
    ax2.plot(eps, history['val_auc'], color='#2ecc71', lw=2.2,
             label='Val Macro AUC (EMA)')
    ax2.set_ylabel('Macro AUC', color='#2ecc71')

    lines  = ax1.get_legend_handles_labels()
    lines2 = ax2.get_legend_handles_labels()
    ax1.legend(lines[0]+lines2[0], lines[1]+lines2[1],
               loc='center right', fontsize=9)
    ax1.set_title(
        f'Model C v2 — Training History  '
        f'(S1={s1_ep}ep · S2={s2_ep}ep · S3={s3_ep}ep · total={total}ep)',
        fontsize=12,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  History  → {save_path}")


# MAIN
def main():
    seed_everything(CFG.SEED)
    os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA not available.\n"
            "In Kaggle: Settings → Accelerator → GPU (H100), then restart kernel."
        )
    device = torch.device('cuda')
    print(f"  GPU : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    # File lists 
    print("\n── Loading file lists ──")
    data_root = Path(CFG.DATA_ROOT)
    if not data_root.exists():
        data_root = auto_discover_root()
    else:
        print(f"  ✓ DATA_ROOT = {data_root}")

    train_paths, train_labels = build_file_list(data_root, 'train')
    test_paths,  test_labels  = build_file_list(data_root, 'val')

    tr_paths, val_paths, tr_labels, val_labels = train_test_split(
        train_paths, train_labels,
        test_size=CFG.VAL_SPLIT, stratify=train_labels,
        random_state=CFG.SEED,
    )
    print(f"\n  Train : {len(tr_paths):,}  |  Val : {len(val_paths):,}  "
          f"|  Test : {len(test_paths):,}\n")

    # DataLoaders 
    train_loader = DataLoader(
        LensDataset(tr_paths,   tr_labels,   get_train_transforms()),
        batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        drop_last=True, persistent_workers=True,
    )
    val_loader = DataLoader(
        LensDataset(val_paths,  val_labels,  get_val_transforms()),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )
    test_loader = DataLoader(
        LensDataset(test_paths, test_labels, get_val_transforms()),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )

    # Model, EMA, loss
    print("\n── Building model ──")
    model     = build_model().to(device)
    ema       = ModelEma(model, decay=CFG.EMA_DECAY, device=device)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    ckpt_path = f"{CFG.OUTPUT_DIR}/best_model.pth"

    history      = {'train_loss': [], 'val_auc': []}
    best_val_auc = 0.0

    def _log(stage_label, ep, total_ep, loss, macro, per_cls, es: EarlyStopping):
        cls_str = '  '.join(f'{k}={v:.4f}' for k, v in per_cls.items())
        print(f"  [{stage_label} {ep:02d}/{total_ep}]  "
              f"loss={loss:.4f}  macro_auc={macro:.4f}  |  {cls_str}{es.status}")

    def _maybe_save(macro, ep_abs):
        nonlocal best_val_auc
        if macro > best_val_auc:
            best_val_auc = macro
            torch.save({'epoch': ep_abs, 'model': ema.ema.state_dict(),
                        'val_auc': macro}, ckpt_path)
            print(f"  ✓  New best val AUC={macro:.4f}  → checkpoint saved")

    # STAGE 1: MLP Head warm-up  (backbone frozen, 12 epochs fixed)
    #
    # Why 12 (not 7 like v1):
    #   v1 logs showed val AUC still climbing at +0.009/ep at ep7 (ended 0.6387).
    #   Model A's single Linear (4.6K params) reached 0.6874 at ep10.
    #   MLPHead has 1187K params — AdamW needs ~12 epochs for momentum
    #   estimates to stabilise across all layers of the MLP before we
    #   hand off to backbone gradients in S2.
    print(f"\n{'═'*62}")
    print(f"  STAGE 1 — MLP Head warm-up  ({CFG.STAGE1_EPOCHS} ep, backbone frozen)")
    print(f"{'═'*62}")

    for name, p in model.named_parameters():
        p.requires_grad = ('head' in name)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable : {trainable/1e3:.1f}K  (MLP head only)")

    s1_opt   = AdamW([p for p in model.parameters() if p.requires_grad],
                     lr=1e-3, weight_decay=CFG.WEIGHT_DECAY)
    s1_sched = cosine_with_warmup(s1_opt, CFG.S1_WARMUP, CFG.STAGE1_EPOCHS)
    es1_dummy = EarlyStopping(patience=999)   # S1 is fixed — dummy ES for logging

    for ep in range(CFG.STAGE1_EPOCHS):
        loss = train_one_epoch(model, train_loader, s1_opt, criterion,
                               s1_sched, ema, device,
                               f'S1 {ep+1:02d}/{CFG.STAGE1_EPOCHS}')
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['train_loss'].append(loss)
        history['val_auc'].append(macro)
        es1_dummy.step(macro)
        _log('S1', ep+1, CFG.STAGE1_EPOCHS, loss, macro, per_cls, es1_dummy)
        _maybe_save(macro, ep)

    print(f"\n  S1 complete → best val AUC: {best_val_auc:.4f}")

    # STAGE 2: Full fine-tuning with LLRD  (max 50 ep, patience 12)
    #
    # Why 50/12 (not 30/8 like v1):
    #   v1 logs: AUC at ep30 = 0.9406, gaining +0.005/ep — nowhere near
    #   plateau. Model A hit 0.9753 at ep40. With a weaker S1 starting point
    #   (0.6387 vs 0.6874), S2 needs to cover ~0.035 more AUC headroom.
    #   Patience 12 = tolerance for ~12 noisy epochs near the ceiling
    #   where gains are ~0.0005/ep.
    print(f"\n{'═'*62}")
    print(f"  STAGE 2 — Full LLRD  "
          f"(max {CFG.STAGE2_EPOCHS} ep · patience={CFG.STAGE2_PATIENCE})")
    print(f"{'═'*62}")

    del s1_opt, s1_sched
    gc.collect(); torch.cuda.empty_cache()
    print(f"  GPU after flush: "
          f"{torch.cuda.memory_allocated()/1e9:.1f} GB alloc / "
          f"{torch.cuda.memory_reserved()/1e9:.1f} GB reserved")

    for p in model.parameters(): p.requires_grad = True

    eff_lr    = CFG.BASE_LR * CFG.BATCH_SIZE / 256    # 3.125e-4
    s2_groups = get_llrd_param_groups(model, eff_lr)
    s2_opt    = AdamW(s2_groups, weight_decay=CFG.WEIGHT_DECAY)
    s2_sched  = cosine_with_warmup(
        s2_opt, CFG.S2_WARMUP, CFG.STAGE2_EPOCHS,
        min_lr_ratio=CFG.MIN_LR / eff_lr,
    )
    es2 = EarlyStopping(patience=CFG.STAGE2_PATIENCE, label='S2')
    actual_s2_epochs = 0

    for ep in range(CFG.STAGE2_EPOCHS):
        abs_ep = CFG.STAGE1_EPOCHS + ep
        loss   = train_one_epoch(model, train_loader, s2_opt, criterion,
                                 s2_sched, ema, device,
                                 f'S2 {ep+1:02d}/{CFG.STAGE2_EPOCHS}')
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['train_loss'].append(loss)
        history['val_auc'].append(macro)
        actual_s2_epochs += 1
        es2.step(macro)
        _log('S2', ep+1, CFG.STAGE2_EPOCHS, loss, macro, per_cls, es2)
        _maybe_save(macro, abs_ep)
        if es2.triggered:
            print(f"\n  ⏹  S2 early stop  ep={ep+1}  "
                  f"best={es2.best:.4f}  "
                  f"(no gain for {CFG.STAGE2_PATIENCE} ep)")
            break

    print(f"\n  S2 complete → {actual_s2_epochs} epochs  |  "
          f"best val AUC: {best_val_auc:.4f}")

    # STAGE 3: Low-LR polish  (max 30 ep, patience 10)
    #
    # Why 30/10 (not 15/6 like v1):
    #   v1 logs: AUC at ep15 = 0.9779, gaining +0.001/ep — still on slope.
    #   Model A's S3 (40 ep) gained +0.0178 total.
    #   30 epochs gives full headroom; patience 10 fires when gains drop
    #   below the noise floor (~+0.0002/ep for 10 consecutive epochs).
    print(f"\n{'═'*62}")
    print(f"  STAGE 3 — Polish  "
          f"(max {CFG.STAGE3_EPOCHS} ep · patience={CFG.STAGE3_PATIENCE} · "
          f"LR={CFG.S3_LR:.2e})")
    print(f"{'═'*62}")

    del s2_opt, s2_sched
    gc.collect(); torch.cuda.empty_cache()

    s3_groups = get_llrd_param_groups(model, CFG.S3_LR)
    s3_opt    = AdamW(s3_groups, weight_decay=CFG.WEIGHT_DECAY)
    s3_sched  = cosine_with_warmup(
        s3_opt, warmup_epochs=0, total_epochs=CFG.STAGE3_EPOCHS,
        min_lr_ratio=0.01,
    )
    es3 = EarlyStopping(patience=CFG.STAGE3_PATIENCE, label='S3')
    actual_s3_epochs = 0

    for ep in range(CFG.STAGE3_EPOCHS):
        abs_ep = CFG.STAGE1_EPOCHS + actual_s2_epochs + ep
        loss   = train_one_epoch(model, train_loader, s3_opt, criterion,
                                 s3_sched, ema, device,
                                 f'S3 {ep+1:02d}/{CFG.STAGE3_EPOCHS}')
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        history['train_loss'].append(loss)
        history['val_auc'].append(macro)
        actual_s3_epochs += 1
        es3.step(macro)
        _log('S3', ep+1, CFG.STAGE3_EPOCHS, loss, macro, per_cls, es3)
        _maybe_save(macro, abs_ep)
        if es3.triggered:
            print(f"\n  ⏹  S3 early stop  ep={ep+1}  "
                  f"best={es3.best:.4f}  "
                  f"(no gain for {CFG.STAGE3_PATIENCE} ep)")
            break

    total_epochs = CFG.STAGE1_EPOCHS + actual_s2_epochs + actual_s3_epochs
    max_budget   = CFG.STAGE1_EPOCHS + CFG.STAGE2_EPOCHS + CFG.STAGE3_EPOCHS
    print(f"\n  S3 complete → {actual_s3_epochs} epochs run")
    print(f"  Total epochs : {total_epochs} / {max_budget} budget  "
          f"(S1={CFG.STAGE1_EPOCHS}, S2={actual_s2_epochs}, S3={actual_s3_epochs})")

    # FINAL EVALUATION
    print(f"\n{'═'*62}")
    print("  FINAL EVALUATION  (best EMA checkpoint)")
    print(f"{'═'*62}")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model'])
    print(f"  Loaded checkpoint from epoch {ckpt['epoch']}  "
          f"(val AUC={ckpt['val_auc']:.4f})")

    #  Single-pass baseline 
    print("\n Single-pass (no TTA)")
    test_auc_plain, test_per_cls_plain, _, _ = evaluate(model, test_loader, device)
    print(f"  Macro AUC (no TTA) : {test_auc_plain:.4f}")
    for cls, val in test_per_cls_plain.items():
        print(f"    {cls:>10} : {val:.4f}")

    #  TTA × 8 
    print(f"\n TTA × {CFG.TTA_N_VIEWS}")
    test_auc_tta, test_per_cls_tta, test_probs_tta, test_lbl = evaluate_tta(
        model, test_paths, test_labels, device, n_views=CFG.TTA_N_VIEWS
    )
    print(f"  Macro AUC (TTA×{CFG.TTA_N_VIEWS})  : {test_auc_tta:.4f}")
    for cls, val in test_per_cls_tta.items():
        print(f"    {cls:>10} : {val:.4f}")

    # Plots
    plot_roc_curves(
        test_lbl, test_probs_tta,
        save_path=f"{CFG.OUTPUT_DIR}/roc_curves_tta.png",
        title=(f"Model C v2 — Test Set  "
               f"(TTA×{CFG.TTA_N_VIEWS}  Macro AUC = {test_auc_tta:.4f})"),
    )
    plot_history(
        history,
        save_path=f"{CFG.OUTPUT_DIR}/training_history.png",
        s1_ep=CFG.STAGE1_EPOCHS,
        s2_ep=actual_s2_epochs,
        s3_ep=actual_s3_epochs,
    )

    # Final summary 
    print(f"\n{'═'*62}")
    print(f"  MODEL C v2 — COMPLETE")
    print(f"{'═'*62}")
    print(f"  Total epochs        : {total_epochs}  "
          f"(S1={CFG.STAGE1_EPOCHS}, S2={actual_s2_epochs}, S3={actual_s3_epochs})")
    print(f"  Best val AUC (EMA)  : {best_val_auc:.4f}")
    print(f"  Test AUC  (no TTA)  : {test_auc_plain:.4f}")
    print(f"  Test AUC  (TTA×{CFG.TTA_N_VIEWS})  : {test_auc_tta:.4f}")
    print(f"  TTA gain            : +{test_auc_tta - test_auc_plain:.4f}")
    print(f"  Checkpoint          : {ckpt_path}")
    print(f"  Outputs             : {CFG.OUTPUT_DIR}")
    print(f"{'═'*62}\n")


if __name__ == '__main__':
    main()

  GPU : NVIDIA H100 80GB HBM3
  VRAM: 85.0 GB

── Loading file lists ──
  ✓ DATA_ROOT = /kaggle/input/datasets/stellarquant/deeplensetask1/dataset
  [train/no_sub]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/no
  [train/subhalo]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/sphere
  [train/vortex]  10,000 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train/vort
  [val/no_sub]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/no
  [val/subhalo]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/sphere
  [val/vortex]  2,500 images  ← /kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val/vort

  Train : 27,000  |  Val : 3,000  |  Test : 7,500


── Building model ──


model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

  Model    : convnextv2_large.fcmae_ft_in22k_in1k_384
  Backbone : 196.4M params
  MLP Head : 1187.3K params  (1536→768→LayerNorm→GELU→Dropout(0.3)→3)
  Total    : 197.6M params  |  grad_ckpt=ON

══════════════════════════════════════════════════════════════
  STAGE 1 — MLP Head warm-up  (12 ep, backbone frozen)
══════════════════════════════════════════════════════════════
  Trainable : 1187.3K  (MLP head only)


S1 01/12:   0%|          | 0/210 [00:00<?, ?it/s]

Eval:   0%|          | 0/12 [00:00<?, ?it/s]

...

  view 5/8:   0%|          | 0/30 [00:00<?, ?it/s]

  view 6/8:   0%|          | 0/30 [00:00<?, ?it/s]

  view 7/8:   0%|          | 0/30 [00:00<?, ?it/s]

  view 8/8:   0%|          | 0/30 [00:00<?, ?it/s]

  Macro AUC (TTA×8)  : 0.9942
        no_sub : 0.9933
       subhalo : 0.9924
        vortex : 0.9969
  ROC plot → /kaggle/working/modelC_v2/roc_curves_tta.png
  History  → /kaggle/working/modelC_v2/training_history.png

══════════════════════════════════════════════════════════════
  MODEL C v2 — COMPLETE
══════════════════════════════════════════════════════════════
  Total epochs        : 92  (S1=12, S2=50, S3=30)
  Best val AUC (EMA)  : 0.9934
  Test AUC  (no TTA)  : 0.9937
  Test AUC  (TTA×8)  : 0.9942
  TTA gain            : +0.0005
  Checkpoint          : /kaggle/working/modelC_v2/best_model.pth
  Outputs             : /kaggle/working/modelC_v2
══════════════════════════════════════════════════════════════

